# 10.10 — Energy-Based Models

Energy-based models (EBMs) describe data by assigning every possible point a scalar **energy**: plausible points should have low energy and implausible points should have high energy. In this lesson, you will build the probability rule, the partition function, Langevin sampling, and contrastive-divergence training from scratch with NumPy so the whole EBM workflow is visible instead of hidden inside a deep-learning library.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build energy-based modeling one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math, including the partition function, the sampling dynamics, and the contrastive update, is derived and shown. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, exponentials, gradients, and tiny simulations.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for samples and toy training.

### 1. Energy turns scores into unnormalized probabilities

An EBM starts with an energy function $E(x)$. Lower energy means the model thinks a state is more plausible, but energy by itself is only a **ranking**. To become a probability, each state receives weight $\exp(-E(x))$, and those weights are divided by the partition function $Z=\sum_x \exp(-E(x))$ in a finite state space.

In [ ]:
states_w = np.array([-1.0, 0.0, 1.0])  # three possible states in a tiny discrete world.
E_w = np.array([0.2, 1.1, 2.0])  # lower energy should become higher probability.
weights_w = np.exp(-E_w)  # unnormalized probability mass for each state.
Z_w = float(np.sum(weights_w))  # partition function: the normalizing constant.
probs_w = weights_w / Z_w  # normalized probabilities that sum to 1.
print("weights:", np.round(weights_w, 4), "Z:", round(Z_w, 3))
print("probabilities:", np.round(probs_w, 4), "sum:", round(float(np.sum(probs_w)), 3))
assert round(Z_w, 3) == 1.287
assert round(float(probs_w[0]), 4) == 0.6362

▶ What you'll see: energies `(0.2, 1.1, 2.0)` become probabilities where the first state has probability about `0.6362`.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["x=-1", "x=0", "x=1"], probs_w, color="teal")
plt.title("1: lower energy becomes higher probability")
plt.ylabel("p(x) = exp(-E) / Z")
plt.show()

▶ What you'll see: the lowest-energy state has the tallest probability bar, but all bars are coupled by the same denominator `Z`.

*Why it's done this way:* exponentiating `-E` makes any real-valued energy into a positive weight, so lower energy creates larger mass. Dividing by `Z` is what turns relative preferences into probabilities; without it, energies can rank states but cannot answer probability questions like “how likely is this state?”

### 2. The partition function couples every state

Changing one energy changes its own weight and also changes $Z$, so every probability moves. This coupling is why EBMs are powerful and hard: the model can define rich landscapes, but exact normalization becomes expensive when there are many or continuous states.

In [ ]:
E_lower_w = E_w.copy()  # start from the previous energies.
E_lower_w[0] -= 0.7  # make the first state even more plausible.
probs_lower_w = np.exp(-E_lower_w) / np.sum(np.exp(-E_lower_w))
print("old probs:", np.round(probs_w, 4))
print("new probs:", np.round(probs_lower_w, 4))
print("probability gained by state 0:", round(float(probs_lower_w[0] - probs_w[0]), 4))
assert probs_lower_w[0] > probs_w[0]
assert probs_lower_w[1] < probs_w[1]

▶ What you'll see: lowering one energy raises its probability and lowers the others because the shared denominator changes.

In [ ]:
shift_w = 5.0  # add the same constant to every energy.
probs_shift_w = np.exp(-(E_w + shift_w)) / np.sum(np.exp(-(E_w + shift_w)))
print("shifted probabilities:", np.round(probs_shift_w, 4))
print("max difference after constant shift:", float(np.max(np.abs(probs_shift_w - probs_w))))
assert np.allclose(probs_shift_w, probs_w)

▶ What you'll see: adding the same offset to all energies changes neither probabilities nor rankings.

*Why it's done this way:* probabilities depend on **energy differences**, not absolute energy offsets. That is why comparing energies across separately trained EBMs is dangerous: one model may add a constant everywhere without changing its distribution at all.

### 3. A continuous energy landscape: wells are modes

For continuous data, $Z=\int \exp(-E(x))dx$ is an integral instead of a sum. We can still inspect a 1-D toy landscape: two low-energy wells should become two high-density modes. A useful EBM energy can be simple or neural; the principle is the same.

In [ ]:
def energy_w(x):  # two-well toy energy, low near -2 and +2.
    x = np.asarray(x)
    return 0.08 * (x**2 - 4.0) ** 2 + 0.15 * x

grid_w = np.linspace(-4, 4, 401)
E_grid_w = energy_w(grid_w)
unnorm_w = np.exp(-E_grid_w)
Z_grid_w = float(np.trapz(unnorm_w, grid_w))  # numerical approximation to the integral.
density_w = unnorm_w / Z_grid_w
print("approx Z:", round(Z_grid_w, 3), "density integral:", round(float(np.trapz(density_w, grid_w)), 3))
assert abs(np.trapz(density_w, grid_w) - 1.0) < 0.01

▶ What you'll see: a numerical partition function normalizes the continuous density over the plotted grid.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(grid_w, E_grid_w, color="purple")
ax[0].set_title("3: energy landscape")
ax[0].set_xlabel("x"); ax[0].set_ylabel("E(x)")
ax[1].plot(grid_w, density_w, color="seagreen")
ax[1].set_title("density proportional to exp(-E)")
ax[1].set_xlabel("x"); ax[1].set_ylabel("p(x)")
plt.show()

▶ What you'll see: two energy valleys line up with two density peaks; look for density concentrating where the energy curve dips.

*Why it's done this way:* plotting energy and density side by side makes the sign convention concrete. High probability is not produced by large positive model output; it is produced by **low** energy after the negative exponential.

### 4. Langevin dynamics samples by noisy downhill motion

If exact sampling is hard, we can move a particle through the landscape. Langevin dynamics uses the update $x_{t+1}=x_t-\eta\nabla_x E(x_t)+\sqrt{2\eta}\,\epsilon_t$. The gradient term moves downhill toward low energy; the noise term prevents the sampler from getting stuck forever in one well.

In [ ]:
def grad_energy_w(x):  # derivative of 0.08(x^2-4)^2 + 0.15x.
    return 0.32 * x * (x**2 - 4.0) + 0.15

x_path_w = [3.5]
eta_w = 0.03
rng_w = np.random.default_rng(1)
for step_w in range(160):
    x_now_w = x_path_w[-1]
    noise_w = np.sqrt(2 * eta_w) * rng_w.normal()
    x_path_w.append(x_now_w - eta_w * grad_energy_w(x_now_w) + noise_w)
x_path_w = np.array(x_path_w)
print("start -> end:", round(float(x_path_w[0]), 3), "->", round(float(x_path_w[-1]), 3))
print("mean last 80 samples:", round(float(np.mean(x_path_w[-80:])), 3))
assert len(x_path_w) == 161

▶ What you'll see: the particle starts far right and then wanders around low-energy regions instead of staying at the initial point.

In [ ]:
plt.figure(figsize=(6, 3))
plt.plot(x_path_w, color="steelblue")
plt.axhline(-2, color="gray", linestyle="--", linewidth=0.8)
plt.axhline(2, color="gray", linestyle="--", linewidth=0.8)
plt.title("4: Langevin path through the energy landscape")
plt.xlabel("step"); plt.ylabel("x")
plt.show()

▶ What you'll see: the path is noisy, but it spends time near the wells around `-2` and `+2` rather than racing monotonically downhill.

*Why it's done this way:* pure gradient descent would collapse into a single local minimum. Langevin adds exactly scaled Gaussian noise so samples can explore a distribution whose density is proportional to `exp(-E)`, making it a sampler rather than just an optimizer.

### 5. Contrastive divergence learns from data and negative samples

Maximum-likelihood training wants data energy to go down and model-sample energy to go up. For an energy $E_\theta(x)$, the gradient has a positive phase from data and a negative phase from samples drawn from the current model. Contrastive divergence approximates the hard model expectation with a few Langevin steps initialized near data.

In [ ]:
rng_cd_w = np.random.default_rng(2)
data_w = np.concatenate([rng_cd_w.normal(-2.0, 0.25, 60), rng_cd_w.normal(2.0, 0.25, 60)])
theta_w = np.array([0.0, 0.0, 0.18])  # E_theta(x)=a*x^4 + b*x^2 + c*x; keep quartic positive.

def energy_theta_w(x, theta):
    return theta[0] * x**4 + theta[1] * x**2 + theta[2] * x

def grad_x_theta_w(x, theta):
    return 4 * theta[0] * x**3 + 2 * theta[1] * x + theta[2]

def features_w(x):
    x = np.asarray(x)
    return np.column_stack([x**4, x**2, x])
print("data mean/std:", round(float(np.mean(data_w)), 3), round(float(np.std(data_w)), 3))
assert data_w.shape == (120,)

▶ What you'll see: synthetic data forms two modes near `-2` and `+2`, the target pattern our energy should make cheap.

In [ ]:
neg_w = data_w.copy() + rng_cd_w.normal(0, 0.2, size=data_w.shape)
for step_cd_w in range(25):
    neg_w = neg_w - 0.02 * grad_x_theta_w(neg_w, theta_w) + np.sqrt(0.04) * rng_cd_w.normal(size=neg_w.shape)

grad_theta_w = np.mean(features_w(data_w), axis=0) - np.mean(features_w(neg_w), axis=0)
print("CD gradient estimate:", np.round(grad_theta_w, 3))
theta_new_w = theta_w - 0.001 * grad_theta_w  # lower data energy, raise negative-sample energy.
theta_new_w[0] = max(theta_new_w[0], 0.002)  # keep the quartic confining for a stable toy energy.
print("theta old -> new:", np.round(theta_w, 4), "->", np.round(theta_new_w, 4))
assert theta_new_w[0] > 0

▶ What you'll see: the gradient compares feature averages under data and short-run negative samples, then nudges the energy parameters.

In [ ]:
E_before_w = energy_theta_w(grid_w, theta_w)
E_after_w = energy_theta_w(grid_w, theta_new_w)
plt.figure(figsize=(6, 3))
plt.hist(data_w, bins=25, density=True, alpha=0.25, color="gray", label="data")
plt.plot(grid_w, E_before_w - np.min(E_before_w), label="energy before", color="red")
plt.plot(grid_w, E_after_w - np.min(E_after_w), label="energy after", color="teal")
plt.title("5: contrastive step reshapes energy")
plt.xlabel("x"); plt.legend(); plt.show()

▶ What you'll see: compare the curves over the data histogram; training aims to carve lower energy where real samples live and higher energy where negatives drift.

*Why it's done this way:* exact likelihood needs samples from the full current model, which requires the intractable partition function or long MCMC. Contrastive divergence uses short negative chains as a practical approximation: it teaches the model to distinguish real data from nearby model-generated alternatives.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, exponentials, gradients, and toy samplers.
import matplotlib.pyplot as plt # load Matplotlib for energy landscapes, histograms, and path plots.
np.random.seed(0) # make stochastic examples reproducible across notebook runs.

def probabilities_from_energy(E): # Convert finite-state energies to normalized probabilities.
    E = np.asarray(E, dtype=float) # Ensure vectorized floating-point arithmetic.
    shifted = E - np.min(E) # Stabilize exponentials without changing probabilities.
    weights = np.exp(-shifted) # Convert lower energies into larger positive weights.
    return weights / np.sum(weights) # Normalize by the finite partition function.

def two_well_energy(x): # Define a reusable double-well toy energy landscape.
    x = np.asarray(x, dtype=float) # Convert scalars or arrays to floating-point values.
    return 0.08 * (x**2 - 4.0) ** 2 + 0.15 * x # Low values near two wells, with a slight tilt.

def two_well_grad(x): # Define the analytic derivative of the double-well energy.
    x = np.asarray(x, dtype=float) # Convert inputs so the derivative broadcasts.
    return 0.32 * x * (x**2 - 4.0) + 0.15 # d/dx of the energy above.

def langevin_1d(x0, grad_fn, steps=100, eta=0.02, seed=0): # Sample with noisy downhill Langevin updates.
    rng = np.random.default_rng(seed) # Use a local generator so examples are reproducible.
    xs = [float(x0)] # Store the full path for plotting and debugging.
    for _ in range(steps): # Take repeated discretized Langevin steps.
        x = xs[-1] # Read the current particle location.
        xs.append(float(x - eta * grad_fn(x) + np.sqrt(2 * eta) * rng.normal())) # Downhill drift plus Gaussian noise.
    return np.array(xs) # Return the path as a NumPy array.

def plot_energy(grid, E, title): # Define a compact energy-curve helper.
    plt.figure(figsize=(5, 3)) # Create a notebook-friendly figure.
    plt.plot(grid, E, color="purple") # Draw energy as a line over the state axis.
    plt.title(title) # Add a descriptive title.
    plt.xlabel("x") # Label the state axis.
    plt.ylabel("energy E(x)") # Label the scalar energy axis.
    plt.show() # Display the plot.

## 🟢 Basics (warm-up)

### Basic 1 — Convert energies to weights

**Goal.** Convert three energies into unnormalized weights, because EBMs begin by using `exp(-E)` as plausibility mass. We build it in 2 steps.

In [ ]:
E_b1 = np.array([0.2, 1.1, 2.0]) # Store finite-state energies from the lesson block.
weights_b1 = np.exp(-E_b1) # Convert energies to positive unnormalized weights.
print("weights:", np.round(weights_b1, 4)) # Inspect how lower energy creates larger weight.
assert round(float(weights_b1[0]), 4) == 0.8187 # Check exp(-0.2).

▶ What you'll see: the first state has the largest weight because its energy is lowest.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact bar plot.
plt.bar(["state0", "state1", "state2"], weights_b1, color="teal") # Visualize unnormalized mass.
plt.title("Basic 1: exp(-energy) weights") # Title the plot.
plt.ylabel("unnormalized weight") # Label the weight scale.
plt.show() # Display the bars.

▶ What you'll see: weights decrease as energy increases.

👀 Takeaway: energy is not probability; `exp(-E)` is the positive weight that starts the normalization process.

### Basic 2 — Normalize with the partition function

**Goal.** Divide by `Z`, because a probability distribution must sum to one. We build it in 2 steps.

In [ ]:
E_b2 = np.array([0.2, 1.1, 2.0]) # Recreate the finite energy vector.
Z_b2 = float(np.sum(np.exp(-E_b2))) # Compute the exact partition function by summing all states.
print("Z:", round(Z_b2, 3)) # Inspect the normalizer.
assert round(Z_b2, 3) == 1.287 # Match the lesson number.

▶ What you'll see: `Z` is the total unnormalized mass across all states.

In [ ]:
p_b2 = np.exp(-E_b2) / Z_b2 # Normalize weights into probabilities.
print("p:", np.round(p_b2, 4), "sum:", round(float(np.sum(p_b2)), 3)) # Inspect normalized values.
plt.figure(figsize=(4, 3)) # Create a compact probability chart.
plt.bar(["state0", "state1", "state2"], p_b2, color="seagreen") # Show probabilities after normalization.
plt.title("Basic 2: probabilities sum to 1") # Title the plot.
plt.ylabel("probability") # Label the probability scale.
plt.show() # Display the plot.

▶ What you'll see: the first probability is `0.6362`, and all probabilities sum to `1`.

👀 Takeaway: the partition function couples all states through one shared denominator.

### Basic 3 — Lower energy raises probability

**Goal.** Change one energy and watch every probability move, because EBMs normalize globally. We build it in 2 steps.

In [ ]:
E_b3 = np.array([0.2, 1.1, 2.0]) # Define the starting energies.
p_old_b3 = probabilities_from_energy(E_b3) # Compute starting probabilities with the helper.
E_new_b3 = E_b3.copy() # Copy energies for a one-state edit.
E_new_b3[0] = -0.5 # Lower the first state's energy.
p_new_b3 = probabilities_from_energy(E_new_b3) # Recompute normalized probabilities.
print("old:", np.round(p_old_b3, 4), "new:", np.round(p_new_b3, 4)) # Compare before and after.
assert p_new_b3[0] > p_old_b3[0] # Lower energy must raise that state's probability.

▶ What you'll see: the first state gains probability while the other states lose probability.

In [ ]:
x_b3 = np.arange(3) # Create positions for grouped bars.
plt.figure(figsize=(5, 3)) # Create a compact comparison plot.
plt.bar(x_b3 - 0.18, p_old_b3, width=0.36, label="old", color="gray") # Plot original probabilities.
plt.bar(x_b3 + 0.18, p_new_b3, width=0.36, label="lower E0", color="teal") # Plot changed probabilities.
plt.xticks(x_b3, ["state0", "state1", "state2"]) # Label states.
plt.title("Basic 3: one energy affects all probabilities") # Title the comparison.
plt.legend() # Show labels.
plt.show() # Display grouped bars.

▶ What you'll see: all bars change even though only one energy was edited.

👀 Takeaway: lowering one energy steals probability mass from the rest through `Z`.

### Basic 4 — Constant energy offsets do not matter

**Goal.** Add the same constant to every energy, because EBMs define probabilities through relative energy differences. We build it in 2 steps.

In [ ]:
E_b4 = np.array([0.2, 1.1, 2.0]) # Define a finite energy vector.
p_b4 = probabilities_from_energy(E_b4) # Compute baseline probabilities.
p_shift_b4 = probabilities_from_energy(E_b4 + 7.0) # Add a constant offset to every energy.
print("max difference:", float(np.max(np.abs(p_b4 - p_shift_b4)))) # Inspect numerical equality.
assert np.allclose(p_b4, p_shift_b4) # A global offset should not change probabilities.

▶ What you'll see: probabilities are unchanged up to floating-point precision.

In [ ]:
plt.figure(figsize=(5, 3)) # Create an energy comparison figure.
plt.plot(E_b4, marker="o", label="E") # Plot original energies.
plt.plot(E_b4 + 7.0, marker="o", label="E + 7") # Plot shifted energies.
plt.title("Basic 4: offsets preserve the distribution") # Title the plot.
plt.ylabel("energy") # Label the energy scale.
plt.legend() # Show curve labels.
plt.show() # Display the line chart.

▶ What you'll see: the shifted curve is higher everywhere, but its probabilities are identical.

👀 Takeaway: absolute energy levels are arbitrary unless a shared normalization context is fixed.

### Basic 5 — Draw a one-dimensional energy landscape

**Goal.** Plot a double-well energy, because EBM intuition comes from seeing low-energy regions as plausible regions. We build it in 2 steps.

In [ ]:
grid_b5 = np.linspace(-4, 4, 301) # Create a 1-D state grid.
E_b5 = two_well_energy(grid_b5) # Evaluate the toy energy at each grid point.
min_x_b5 = float(grid_b5[np.argmin(E_b5)]) # Find the lowest plotted energy location.
print("minimum near x:", round(min_x_b5, 2)) # Inspect one low-energy well.
assert -3.0 < min_x_b5 < 3.0 # Check the minimum is inside the plotted domain.

▶ What you'll see: the lowest point is near one of the two wells.

In [ ]:
plot_energy(grid_b5, E_b5, "Basic 5: double-well EBM energy") # Visualize the reusable toy landscape.

▶ What you'll see: two valleys separated by a higher-energy barrier.

👀 Takeaway: an EBM distribution is shaped by valleys and barriers in the energy landscape.

### Basic 6 — Approximate continuous normalization

**Goal.** Numerically integrate `exp(-E)` on a grid, because continuous EBMs use an integral partition function. We build it in 3 steps.

In [ ]:
grid_b6 = np.linspace(-4, 4, 401) # Create a fine grid for numerical integration.
E_b6 = two_well_energy(grid_b6) # Evaluate the continuous toy energy.
weights_b6 = np.exp(-E_b6) # Convert energy to unnormalized density values.
print("weight range:", round(float(np.min(weights_b6)), 4), round(float(np.max(weights_b6)), 4)) # Inspect positive masses.

▶ What you'll see: density weights are largest near low-energy wells.

In [ ]:
Z_b6 = float(np.trapz(weights_b6, grid_b6)) # Approximate the continuous partition integral.
density_b6 = weights_b6 / Z_b6 # Normalize to an approximate density on the grid.
print("Z approx:", round(Z_b6, 3), "area:", round(float(np.trapz(density_b6, grid_b6)), 3)) # Inspect normalization.
assert abs(np.trapz(density_b6, grid_b6) - 1.0) < 0.01 # Check density integrates to 1 approximately.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a compact density plot.
plt.plot(grid_b6, density_b6, color="seagreen") # Plot normalized approximate density.
plt.title("Basic 6: density proportional to exp(-E)") # Title the plot.
plt.xlabel("x") # Label state axis.
plt.ylabel("approx p(x)") # Label density axis.
plt.show() # Display density.

▶ What you'll see: density peaks sit where the energy landscape has wells.

👀 Takeaway: continuous `Z` is an integral, not a simple sum, and is usually the hard part.

### Basic 7 — Compute the energy gradient

**Goal.** Evaluate `dE/dx`, because samplers need the direction of steepest energy increase so they can step downhill. We build it in 2 steps.

In [ ]:
points_b7 = np.array([-3.0, -2.0, 0.0, 2.0, 3.0]) # Choose inspectable locations.
grads_b7 = two_well_grad(points_b7) # Compute analytic energy gradients.
print("gradients:", np.round(grads_b7, 3)) # Inspect signs and magnitudes.
assert grads_b7[0] < 0 and grads_b7[-1] > 0 # Tails push back toward the middle wells.

▶ What you'll see: gradients point outward in sign, so negative gradients pull the tails inward.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a gradient-vector plot.
plt.axhline(0, color="black", linewidth=0.8) # Add a zero-gradient reference.
plt.plot(points_b7, grads_b7, marker="o", color="orange") # Plot gradient values.
plt.title("Basic 7: energy gradient") # Title the plot.
plt.xlabel("x") # Label state axis.
plt.ylabel("dE/dx") # Label derivative axis.
plt.show() # Display gradient plot.

▶ What you'll see: zero crossings occur near stationary points of the energy curve.

👀 Takeaway: negative gradient is the deterministic downhill direction used by Langevin sampling.

### Basic 8 — Take one Langevin step

**Goal.** Combine downhill drift with Gaussian noise, because Langevin sampling must both seek low energy and explore. We build it in 2 steps.

In [ ]:
x_b8 = 3.0 # Start in the right tail of the landscape.
eta_b8 = 0.05 # Choose a small step size.
grad_b8 = float(two_well_grad(x_b8)) # Compute local energy gradient.
drift_b8 = -eta_b8 * grad_b8 # Convert gradient to downhill drift.
print("gradient:", round(grad_b8, 3), "drift:", round(drift_b8, 3)) # Inspect deterministic motion.
assert drift_b8 < 0 # From x=3, downhill motion should move left.

▶ What you'll see: the deterministic part pulls the particle left toward lower energy.

In [ ]:
noise_b8 = np.sqrt(2 * eta_b8) * 0.25 # Use a fixed standard-normal draw of 0.25 for auditability.
x_next_b8 = x_b8 + drift_b8 + noise_b8 # Apply one Langevin update.
print("noise:", round(noise_b8, 3), "next x:", round(x_next_b8, 3)) # Inspect the full step.
assert round(x_next_b8, 3) == 2.832 # Check the concrete one-step calculation.

In [ ]:
parts_b8 = [x_b8, drift_b8, noise_b8, x_next_b8] # Reuse the audited one-step quantities.
labels_b8 = ["start", "drift", "noise", "next"] # Name each contribution.
colors_b8 = ["gray", "tomato", "goldenrod", "steelblue"] # Highlight deterministic and random pieces.
plt.figure(figsize=(5, 3)) # Create a compact contribution plot.
plt.bar(labels_b8, parts_b8, color=colors_b8) # Compare the start, update pieces, and result.
plt.axhline(0, color="black", linewidth=0.8) # Mark zero for signed drift/noise.
plt.title("Basic 8: one Langevin step") # Title the update decomposition.
plt.ylabel("value") # Label value axis.
plt.show() # Display the one-step breakdown.

▶ What you'll see: the step moves left overall, with a small random kick.

👀 Takeaway: Langevin dynamics is not plain optimization; noise is part of the sampling rule.

### Basic 9 — Run a short Langevin chain

**Goal.** Simulate many Langevin steps, because an EBM sample is produced by iterating local updates. We build it in 2 steps.

In [ ]:
path_b9 = langevin_1d(3.5, two_well_grad, steps=120, eta=0.03, seed=9) # Run a reproducible 1-D chain.
print("path length:", len(path_b9), "final x:", round(float(path_b9[-1]), 3)) # Inspect the chain output.
assert len(path_b9) == 121 # Check steps plus the starting state.

▶ What you'll see: the chain returns every intermediate position, not just the final sample.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a path plot.
plt.plot(path_b9, color="steelblue") # Draw position over time.
plt.title("Basic 9: short Langevin chain") # Title the path.
plt.xlabel("step") # Label iteration axis.
plt.ylabel("x") # Label state value.
plt.show() # Display chain.

▶ What you'll see: the particle wanders rather than following a perfectly smooth path.

👀 Takeaway: repeated noisy downhill moves create approximate samples from the energy-defined distribution.

### Basic 10 — Compare data and negative samples

**Goal.** Put real data beside model negatives, because contrastive learning depends on their difference. We build it in 3 steps.

In [ ]:
rng_b10 = np.random.default_rng(10) # Create reproducible randomness.
data_b10 = np.concatenate([rng_b10.normal(-2, 0.2, 40), rng_b10.normal(2, 0.2, 40)]) # Synthesize two-mode data.
neg_b10 = rng_b10.normal(0, 1.5, 80) # Create crude negative samples from a broad proposal.
print("data mean:", round(float(np.mean(data_b10)), 3), "negative mean:", round(float(np.mean(neg_b10)), 3)) # Inspect summaries.
assert data_b10.shape == neg_b10.shape # Check equal batch sizes.

▶ What you'll see: data is bimodal around ±2 while negatives are broader around 0.

In [ ]:
feat_data_b10 = np.mean(data_b10 ** 2) # Use a simple feature expectation under data.
feat_neg_b10 = np.mean(neg_b10 ** 2) # Use the same feature under negatives.
print("E_data[x^2]:", round(float(feat_data_b10), 3), "E_neg[x^2]:", round(float(feat_neg_b10), 3)) # Compare expectations.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a histogram comparison.
plt.hist(data_b10, bins=20, alpha=0.55, density=True, label="data", color="teal") # Plot data density.
plt.hist(neg_b10, bins=20, alpha=0.45, density=True, label="negative", color="red") # Plot negative density.
plt.title("Basic 10: data vs negative samples") # Title the contrast.
plt.legend() # Show labels.
plt.show() # Display histograms.

▶ What you'll see: data piles near the two wells; negatives include locations the model should learn to make higher energy.

👀 Takeaway: contrastive methods train from the gap between observed data and model-generated alternatives.

## 🟡 Easy

### Easy 1 — Implement a finite-state EBM

**Goal.** Build a complete tiny EBM probability table, because finite states let us audit `E`, `exp(-E)`, `Z`, and `p` exactly. We build it in 3 steps.

In [ ]:
states_e1 = np.array(["cat", "dog", "car", "rock"]) # Name four finite states.
E_e1 = np.array([0.1, 0.4, 1.6, 2.2]) # Assign lower energies to more plausible states.
print("states:", states_e1) # Inspect labels.
print("energies:", E_e1) # Inspect scalar energy values.

▶ What you'll see: each possible state has exactly one scalar energy.

In [ ]:
p_e1 = probabilities_from_energy(E_e1) # Normalize finite-state energies.
print("probabilities:", np.round(p_e1, 4), "sum:", round(float(np.sum(p_e1)), 3)) # Inspect distribution.
assert round(float(np.sum(p_e1)), 3) == 1.0 # Verify normalization.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a probability table plot.
plt.bar(states_e1, p_e1, color="seagreen") # Plot normalized probabilities.
plt.title("Easy 1: finite-state EBM distribution") # Title the plot.
plt.ylabel("p(state)") # Label probability scale.
plt.show() # Display bars.

▶ What you'll see: plausible labels have taller bars because their energies are lower.

👀 Takeaway: a finite EBM is a normalized table built from energies; the trouble starts when the table becomes huge.

### Easy 2 — Visualize an energy landscape and density together

**Goal.** Plot `E(x)` and normalized `p(x)` on the same grid, because the sign relationship is easy to reverse mentally. We build it in 3 steps.

In [ ]:
grid_e2 = np.linspace(-4, 4, 500) # Create a dense plotting grid.
E_e2 = two_well_energy(grid_e2) # Evaluate energy on the grid.
weights_e2 = np.exp(-E_e2) # Convert energy to unnormalized density.
Z_e2 = float(np.trapz(weights_e2, grid_e2)) # Approximate continuous Z.
p_e2 = weights_e2 / Z_e2 # Normalize the grid density.
print("Z approx:", round(Z_e2, 3)) # Inspect normalizer.

▶ What you'll see: continuous normalization uses numerical integration in this toy example.

In [ ]:
mode_locations_e2 = grid_e2[np.argsort(p_e2)[-2:]] # Read two high-density grid locations.
print("two high-density grid points:", np.round(mode_locations_e2, 2)) # Inspect approximate modes.
assert np.max(p_e2) > np.median(p_e2) # Density should peak somewhere.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 3)) # Create side-by-side plots.
ax[0].plot(grid_e2, E_e2, color="purple") # Draw energy.
ax[0].set_title("energy E(x)") # Title energy panel.
ax[1].plot(grid_e2, p_e2, color="teal") # Draw normalized density.
ax[1].set_title("density ∝ exp(-E)") # Title density panel.
plt.suptitle("Easy 2: valleys become modes") # Overall title.
plt.show() # Display panels.

▶ What you'll see: density peaks appear over energy valleys; look for the mirror-like relationship.

👀 Takeaway: EBMs model data by carving low-energy basins where probability should concentrate.

### Easy 3 — Sample with Langevin dynamics

**Goal.** Generate approximate samples from the toy energy, because EBMs often rely on MCMC rather than direct decoding. We build it in 4 steps.

In [ ]:
seeds_e3 = np.arange(12) # Use multiple independent chains.
starts_e3 = np.linspace(-3.5, 3.5, len(seeds_e3)) # Spread starting points across the landscape.
print("starts:", np.round(starts_e3, 2)) # Inspect initial states.

▶ What you'll see: chains begin on both sides of the energy landscape.

In [ ]:
paths_e3 = np.array([langevin_1d(starts_e3[i], two_well_grad, steps=180, eta=0.025, seed=int(seeds_e3[i])) for i in range(len(seeds_e3))]) # Run all chains.
finals_e3 = paths_e3[:, -1] # Take final states as approximate samples.
print("final sample mean/std:", round(float(np.mean(finals_e3)), 3), round(float(np.std(finals_e3)), 3)) # Inspect sample spread.
assert paths_e3.shape == (12, 181) # Verify chains and steps.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a multi-chain path plot.
for i_e3 in range(paths_e3.shape[0]): # Plot every chain lightly.
    plt.plot(paths_e3[i_e3], alpha=0.55) # Draw one trajectory.
plt.title("Easy 3: multiple Langevin chains") # Title trajectories.
plt.xlabel("step") # Label time.
plt.ylabel("x") # Label state.
plt.show() # Display paths.

▶ What you'll see: different chains wander into low-energy regions, sometimes from different starting sides.

In [ ]:
plt.figure(figsize=(5, 3)) # Create final-sample histogram.
plt.hist(finals_e3, bins=8, color="teal", alpha=0.75) # Show approximate samples.
plt.title("Easy 3: final Langevin samples") # Title histogram.
plt.xlabel("x") # Label sample values.
plt.show() # Display histogram.

▶ What you'll see: final samples tend to land near the energy wells rather than uniformly across the line.

👀 Takeaway: Langevin sampling converts an energy gradient into generated examples by iterative noisy motion.

### Easy 4 — Train a quadratic energy by contrastive divergence

**Goal.** Fit a simple Gaussian-like energy to one cluster, because CD can be understood before using neural networks. We build it in 4 steps.

In [ ]:
rng_e4 = np.random.default_rng(4) # Create reproducible randomness.
data_e4 = rng_e4.normal(1.5, 0.35, size=100) # Synthesize one-cluster data.
mu_e4 = 0.0 # Initialize energy center E=(x-mu)^2/(2s^2).
sigma_e4 = 1.0 # Use fixed scale for the toy energy.
print("data mean:", round(float(np.mean(data_e4)), 3)) # Inspect target center.

▶ What you'll see: the data center is near `1.5`, while the initial energy center is `0`.

In [ ]:
for epoch_e4 in range(60): # Run simple CD-style updates.
    neg_e4 = data_e4 + rng_e4.normal(0, 0.4, size=data_e4.shape) # Initialize negatives near data.
    for _ in range(8): # Take short Langevin chains under current quadratic energy.
        grad_neg_e4 = (neg_e4 - mu_e4) / (sigma_e4**2) # dE/dx for the quadratic energy.
        neg_e4 = neg_e4 - 0.04 * grad_neg_e4 + np.sqrt(0.08) * rng_e4.normal(size=neg_e4.shape) # Langevin step.
    grad_mu_e4 = np.mean(mu_e4 - data_e4) - np.mean(mu_e4 - neg_e4) # CD gradient wrt mu, up to fixed scale.
    mu_e4 -= 0.1 * grad_mu_e4 # Move center to lower data energy relative to negatives.
print("learned mu:", round(float(mu_e4), 3)) # Inspect fitted center.
assert abs(mu_e4 - np.mean(data_e4)) < 0.5 # Check it moved near data.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
grid_e4 = np.linspace(-1, 3.5, 250) # Create plotting grid.
E_e4 = (grid_e4 - mu_e4) ** 2 / (2 * sigma_e4**2) # Evaluate learned quadratic energy.
print("energy at learned mu:", round(float((mu_e4 - mu_e4) ** 2), 3)) # Verify minimum energy location.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
plt.figure(figsize=(5, 3)) # Create histogram plus energy plot.
plt.hist(data_e4, bins=18, density=True, alpha=0.35, color="gray", label="data") # Plot data.
plt.plot(grid_e4, E_e4, color="purple", label="learned energy") # Plot energy curve.
plt.axvline(mu_e4, color="teal", linestyle="--", label="mu") # Mark learned center.
plt.title("Easy 4: CD learns a low-energy center") # Title plot.
plt.legend() # Show labels.
plt.show() # Display figure.

▶ What you'll see: the energy minimum shifts close to the data cluster center.

👀 Takeaway: contrastive divergence lowers energy at data relative to short-run negative samples.

### Easy 5 — Show weak negatives create weak learning

**Goal.** Compare nearby and broad negative samples, because an EBM only learns to raise energy where the sampler visits. We build it in 3 steps.

In [ ]:
rng_e5 = np.random.default_rng(5) # Create reproducible randomness.
data_e5 = rng_e5.normal(0.0, 0.2, size=120) # Data concentrated near zero.
near_neg_e5 = data_e5 + rng_e5.normal(0.0, 0.05, size=data_e5.shape) # Weak negatives almost identical to data.
far_neg_e5 = rng_e5.normal(1.5, 0.4, size=data_e5.shape) # Stronger negatives in a different region.
print("near mean:", round(float(np.mean(near_neg_e5)), 3), "far mean:", round(float(np.mean(far_neg_e5)), 3)) # Inspect contrast.

▶ What you'll see: nearby negatives are barely distinguishable from data, while broad negatives occupy another region.

In [ ]:
contrast_near_e5 = abs(float(np.mean(data_e5**2) - np.mean(near_neg_e5**2))) # Feature gap for weak negatives.
contrast_far_e5 = abs(float(np.mean(data_e5**2) - np.mean(far_neg_e5**2))) # Feature gap for stronger negatives.
print("feature contrast near/far:", round(contrast_near_e5, 4), round(contrast_far_e5, 4)) # Compare training signals.
assert contrast_far_e5 > contrast_near_e5 # Stronger negatives should create larger signal.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
plt.figure(figsize=(5, 3)) # Create contrast histogram.
plt.hist(data_e5, bins=20, density=True, alpha=0.45, label="data", color="teal") # Plot data.
plt.hist(near_neg_e5, bins=20, density=True, alpha=0.35, label="near negatives", color="orange") # Plot weak negatives.
plt.hist(far_neg_e5, bins=20, density=True, alpha=0.30, label="far negatives", color="red") # Plot stronger negatives.
plt.title("Easy 5: negative sample quality") # Title plot.
plt.legend() # Show labels.
plt.show() # Display histograms.

▶ What you'll see: weak negatives overlap data so much that the contrastive signal is tiny.

👀 Takeaway: contrastive divergence is only as informative as the negative samples it generates.

## 🔴 Advanced

### Advanced 1 — Estimate a partition function by grid resolution

**Goal.** Compare grid resolutions for continuous `Z`, because numerical normalization can be sensitive when the landscape has narrow regions. We build it in 4 steps.

In [ ]:
resolutions_a1 = np.array([51, 101, 201, 401, 801]) # Try increasingly fine grids.
Zs_a1 = [] # Store partition estimates.
print("resolutions:", resolutions_a1) # Inspect grid sizes.

▶ What you'll see: the experiment tests coarse to fine quadrature.

In [ ]:
for n_a1 in resolutions_a1: # Loop over grid sizes.
    grid_a1 = np.linspace(-4, 4, int(n_a1)) # Build the current grid.
    Z_a1 = float(np.trapz(np.exp(-two_well_energy(grid_a1)), grid_a1)) # Approximate continuous Z.
    Zs_a1.append(Z_a1) # Store estimate.
print("Z estimates:", np.round(Zs_a1, 4)) # Inspect convergence.
assert abs(Zs_a1[-1] - Zs_a1[-2]) < 0.02 # Check fine grids agree.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
changes_a1 = np.abs(np.diff(Zs_a1)) # Measure successive changes.
print("successive changes:", np.round(changes_a1, 5)) # Inspect convergence rate.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
plt.figure(figsize=(5, 3)) # Create convergence plot.
plt.plot(resolutions_a1, Zs_a1, marker="o", color="purple") # Plot Z estimate by grid size.
plt.title("Advanced 1: partition estimate convergence") # Title plot.
plt.xlabel("grid points") # Label resolution.
plt.ylabel("estimated Z") # Label partition estimate.
plt.show() # Display curve.

▶ What you'll see: estimates stabilize as the grid gets fine enough to resolve the energy wells.

👀 Takeaway: exact `Z` is easy only in tiny examples; real EBMs need approximations or avoid computing it directly.

### Advanced 2 — Compare Langevin step sizes

**Goal.** Show stable and unstable sampling, because too-large Langevin steps discretize the dynamics poorly. We build it in 4 steps.

In [ ]:
etas_a2 = np.array([0.01, 0.04, 0.18]) # Compare small, moderate, and aggressive steps.
paths_a2 = [] # Store one chain per step size.
print("etas:", etas_a2) # Inspect step sizes.

▶ What you'll see: the largest step is intentionally risky for the toy sampler.

In [ ]:
for i_a2, eta_a2 in enumerate(etas_a2): # Run each step size.
    path_a2 = langevin_1d(3.2, two_well_grad, steps=160, eta=float(eta_a2), seed=20 + i_a2) # Simulate chain.
    paths_a2.append(path_a2) # Store path.
finals_a2 = np.array([p[-1] for p in paths_a2]) # Gather final states.
print("finals:", np.round(finals_a2, 3)) # Inspect endpoints.
assert len(paths_a2) == 3 # Check all chains ran.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
ranges_a2 = np.array([np.max(p) - np.min(p) for p in paths_a2]) # Measure path spread.
print("path ranges:", np.round(ranges_a2, 3)) # Inspect motion scale.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
plt.figure(figsize=(6, 3)) # Create step-size comparison plot.
for eta_a2, path_a2 in zip(etas_a2, paths_a2): # Plot each path.
    plt.plot(path_a2, label=f"eta={eta_a2}") # Draw trajectory.
plt.title("Advanced 2: Langevin step-size effects") # Title plot.
plt.xlabel("step") # Label time.
plt.ylabel("x") # Label state.
plt.legend() # Show labels.
plt.show() # Display paths.

▶ What you'll see: larger steps move more aggressively and can jump across the landscape instead of smoothly exploring it.

👀 Takeaway: Langevin step size is a sampling hyperparameter, not just a speed knob.

### Advanced 3 — Train a two-well polynomial EBM

**Goal.** Learn a quartic energy for bimodal data with CD, because a quadratic energy cannot represent two separated modes. We build it in 5 steps.

In [ ]:
rng_a3 = np.random.default_rng(30) # Create reproducible randomness.
data_a3 = np.concatenate([rng_a3.normal(-2, 0.25, 100), rng_a3.normal(2, 0.25, 100)]) # Bimodal data.
theta_a3 = np.array([0.02, 0.0, 0.0]) # E=a*x^4 + b*x^2 + c*x with positive quartic.
print("initial theta:", theta_a3) # Inspect starting parameters.

▶ What you'll see: the model begins with a shallow symmetric quartic.

In [ ]:
def energy_poly_a3(x, theta): # Define parameterized polynomial energy.
    return theta[0] * x**4 + theta[1] * x**2 + theta[2] * x # Compute scalar energy.

def grad_poly_x_a3(x, theta): # Define derivative with respect to x.
    return 4 * theta[0] * x**3 + 2 * theta[1] * x + theta[2] # Analytic gradient for Langevin.

def feat_poly_a3(x): # Define features whose weights are theta.
    x = np.asarray(x) # Convert to array.
    return np.column_stack([x**4, x**2, x]) # Return feature matrix.
print("feature shape:", feat_poly_a3(data_a3[:3]).shape) # Inspect feature dimensions.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
loss_proxy_a3 = [] # Track data-vs-negative energy gap.
for epoch_a3 in range(80): # Run short CD training.
    neg_a3 = rng_a3.normal(0, 2.5, size=data_a3.shape) # Initialize negatives broadly.
    for _ in range(20): # Short-run Langevin under current energy.
        neg_a3 = neg_a3 - 0.015 * grad_poly_x_a3(neg_a3, theta_a3) + np.sqrt(0.03) * rng_a3.normal(size=neg_a3.shape) # Langevin step.
        neg_a3 = np.clip(neg_a3, -5, 5) # Keep toy chains finite.
    grad_theta_a3 = np.mean(feat_poly_a3(data_a3), axis=0) - np.mean(feat_poly_a3(neg_a3), axis=0) # CD gradient.
    theta_a3 -= 0.0005 * grad_theta_a3 # Lower data energy relative to negatives.
    theta_a3[0] = max(theta_a3[0], 0.005) # Keep energy confining.
    loss_proxy_a3.append(float(np.mean(energy_poly_a3(data_a3, theta_a3)) - np.mean(energy_poly_a3(neg_a3, theta_a3)))) # Track contrast.
print("trained theta:", np.round(theta_a3, 4)) # Inspect learned parameters.
assert theta_a3[0] > 0 # Verify confining quartic remains positive.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
grid_a3 = np.linspace(-4, 4, 401) # Create plotting grid.
E_a3 = energy_poly_a3(grid_a3, theta_a3) # Evaluate trained energy.
left_E_a3 = float(energy_poly_a3(-2.0, theta_a3)) # Energy near left data mode.
center_E_a3 = float(energy_poly_a3(0.0, theta_a3)) # Energy near middle gap.
right_E_a3 = float(energy_poly_a3(2.0, theta_a3)) # Energy near right data mode.
print("E(-2), E(0), E(2):", np.round([left_E_a3, center_E_a3, right_E_a3], 3)) # Inspect mode energies.
assert np.isfinite(E_a3).all() # Ensure the learned curve is finite.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
plt.figure(figsize=(6, 3)) # Create trained energy plot.
plt.hist(data_a3, bins=30, density=True, alpha=0.3, color="gray", label="data") # Show data modes.
plt.plot(grid_a3, E_a3 - np.min(E_a3), color="purple", label="learned energy") # Plot shifted energy.
plt.title("Advanced 3: quartic EBM after CD") # Title plot.
plt.xlabel("x") # Label state.
plt.legend() # Show labels.
plt.show() # Display plot.

▶ What you'll see: the learned energy is low around the two data clusters and higher away from them; inspect whether the middle barrier is visible.

👀 Takeaway: richer energy functions can represent multimodal data, but their negative-sample training is more delicate.

### Advanced 4 — Demonstrate arbitrary energy offsets in model comparison

**Goal.** Compare two equivalent shifted energies, because raw energy numbers are not calibrated across separate EBMs. We build it in 3 steps.

In [ ]:
grid_a4 = np.linspace(-3, 3, 301) # Create a comparison grid.
E1_a4 = 0.5 * (grid_a4 - 1.0) ** 2 # Define one quadratic energy.
E2_a4 = E1_a4 + 12.0 # Add a large constant offset.
p1_a4 = np.exp(-E1_a4) / np.trapz(np.exp(-E1_a4), grid_a4) # Normalize first density.
p2_a4 = np.exp(-E2_a4) / np.trapz(np.exp(-E2_a4), grid_a4) # Normalize shifted density.
print("max density difference:", float(np.max(np.abs(p1_a4 - p2_a4)))) # Inspect equivalence.
assert np.allclose(p1_a4, p2_a4) # Shifted energies define same density.

▶ What you'll see: the densities match even though one energy curve is 12 units higher.

In [ ]:
print("raw E1 at x=1:", round(float(np.min(E1_a4)), 3), "raw E2 at x=1:", round(float(np.min(E2_a4)), 3)) # Compare raw energies.
print("same argmin:", round(float(grid_a4[np.argmin(E1_a4)]), 2), round(float(grid_a4[np.argmin(E2_a4)]), 2)) # Compare minima.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
plt.figure(figsize=(6, 3)) # Create density comparison plot.
plt.plot(grid_a4, p1_a4, label="E", color="teal") # Plot first normalized density.
plt.plot(grid_a4, p2_a4, linestyle="--", label="E + 12", color="red") # Plot shifted normalized density.
plt.title("Advanced 4: shifted energies, same probabilities") # Title plot.
plt.xlabel("x") # Label state.
plt.ylabel("normalized density") # Label density.
plt.legend() # Show labels.
plt.show() # Display plot.

▶ What you'll see: the two density curves overlap, proving the raw offset carried no probabilistic meaning.

👀 Takeaway: compare energies inside one model, but avoid treating raw energies from different models as calibrated scores.

### Advanced 5 — Use a tiny persistent contrastive-divergence buffer

**Goal.** Keep negative particles across updates, because persistent CD often gives stronger negatives than restarting chains from data every time. We build it in 5 steps.

In [ ]:
rng_a5 = np.random.default_rng(50) # Create reproducible randomness.
data_a5 = np.concatenate([rng_a5.normal(-1.5, 0.2, 80), rng_a5.normal(1.5, 0.2, 80)]) # Create two-mode data.
particles_a5 = rng_a5.normal(0, 2.0, size=80) # Initialize a persistent negative buffer.
theta_a5 = np.array([0.03, -0.08, 0.0]) # Start with a confining quartic and a mild double-well term.
print("particles shape:", particles_a5.shape) # Inspect buffer size.

▶ What you'll see: the model keeps a reusable set of negative particles.

In [ ]:
def energy_a5(x, theta): # Define persistent-CD energy.
    return theta[0] * x**4 + theta[1] * x**2 + theta[2] * x # Polynomial energy.

def grad_x_a5(x, theta): # Define x-gradient for sampling.
    return 4 * theta[0] * x**3 + 2 * theta[1] * x + theta[2] # Analytic derivative.

def feat_a5(x): # Define parameter features.
    x = np.asarray(x) # Convert input.
    return np.column_stack([x**4, x**2, x]) # Feature matrix.
print("initial data energy:", round(float(np.mean(energy_a5(data_a5, theta_a5))), 3)) # Inspect starting data energy.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
energy_gaps_a5 = [] # Track data energy minus particle energy.
for epoch_a5 in range(70): # Run persistent CD updates.
    for _ in range(12): # Refresh persistent particles with short Langevin runs.
        particles_a5 = particles_a5 - 0.02 * grad_x_a5(particles_a5, theta_a5) + np.sqrt(0.04) * rng_a5.normal(size=particles_a5.shape) # PCD sampler step.
        particles_a5 = np.clip(particles_a5, -4, 4) # Keep toy particles bounded.
    batch_a5 = rng_a5.choice(data_a5, size=particles_a5.shape[0], replace=False) # Match data batch size.
    grad_theta_a5 = np.mean(feat_a5(batch_a5), axis=0) - np.mean(feat_a5(particles_a5), axis=0) # Contrast data and persistent negatives.
    theta_a5 -= 0.0008 * grad_theta_a5 # Apply CD update.
    theta_a5[0] = max(theta_a5[0], 0.006) # Maintain confining positive quartic.
    energy_gaps_a5.append(float(np.mean(energy_a5(batch_a5, theta_a5)) - np.mean(energy_a5(particles_a5, theta_a5)))) # Store contrast.
print("final theta:", np.round(theta_a5, 4)) # Inspect trained parameters.
assert np.isfinite(theta_a5).all() # Check numerical stability.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
print("particle mean/std:", round(float(np.mean(particles_a5)), 3), round(float(np.std(particles_a5)), 3)) # Inspect persistent negatives.
print("last energy gap:", round(float(energy_gaps_a5[-1]), 3)) # Inspect final contrast.
assert len(energy_gaps_a5) == 70 # Confirm all epochs recorded.

▶ What you'll see: the printed diagnostics make this intermediate calculation auditable before the next step.

In [ ]:
grid_a5 = np.linspace(-4, 4, 401) # Create plotting grid.
plt.figure(figsize=(6, 3)) # Create PCD diagnostic plot.
plt.hist(data_a5, bins=25, density=True, alpha=0.30, label="data", color="gray") # Plot real data.
plt.hist(particles_a5, bins=25, density=True, alpha=0.30, label="persistent negatives", color="red") # Plot particles.
plt.plot(grid_a5, energy_a5(grid_a5, theta_a5) - np.min(energy_a5(grid_a5, theta_a5)), color="purple", label="shifted energy") # Plot learned energy.
plt.title("Advanced 5: persistent contrastive divergence") # Title diagnostic.
plt.legend() # Show labels.
plt.show() # Display plot.

▶ What you'll see: persistent negatives occupy regions the current model samples, giving a more model-aware contrast than fresh random negatives.

👀 Takeaway: persistent CD improves negative-sample pressure by letting sampler particles track the model as it changes.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

An energy-based model learns what should be low energy, then samples by moving downhill through that landscape.

RBMs are the discrete ancestor, while gradients and MCMC explain how samples move through an energy landscape. Score models reuse the same idea locally by learning gradients of log density.

Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

SEED = 1008
rng = np.random.default_rng(SEED)


def standardize_features(X):
    X = np.asarray(X, dtype=float)
    mu = X.mean(axis=0, keepdims=True)
    sd = X.std(axis=0, keepdims=True) + 1e-6
    return (X - mu) / sd


def pca_features(X, n_components=8):
    X = standardize_features(X)
    k = min(n_components, X.shape[1], max(1, X.shape[0] - 1))
    if k == X.shape[1]:
        return X
    pca = PCA(n_components=k, random_state=SEED)
    return pca.fit_transform(X)


def make_f9_ladder(seed=SEED):
    local = np.random.default_rng(seed)
    rungs = []

    x1 = local.normal(0.0, 1.0, size=(240, 1))
    y1 = (x1[:, 0] > 0).astype(int)
    rungs.append({"name": "D1 1-D Gaussian", "X": x1, "y": y1, "kind": "points"})

    x2, y2 = make_moons(n_samples=320, noise=0.08, random_state=seed + 2)
    rungs.append({"name": "D2 2-D moons", "X": x2, "y": y2, "kind": "points"})

    means = np.array([[-2.0, -1.0], [1.8, -0.6], [0.2, 2.0]])
    covs = np.array([[[0.20, 0.05], [0.05, 0.45]], [[0.45, -0.18], [-0.18, 0.25]], [[0.25, 0.0], [0.0, 0.25]]])
    parts = []
    labels = []
    for k, (mean, cov) in enumerate(zip(means, covs)):
        pts = local.multivariate_normal(mean, cov, size=120)
        parts.append(pts)
        labels.append(np.full(120, k))
    x3 = np.vstack(parts)
    y3 = np.concatenate(labels)
    rungs.append({"name": "D3 three-component mixture", "X": x3, "y": y3, "kind": "points"})

    digits = load_digits()
    x4 = digits.data.astype(float) / 16.0
    y4 = digits.target.astype(int)
    rungs.append({"name": "D4 sklearn digits", "X": x4[:900], "y": y4[:900], "kind": "image", "image_shape": (8, 8)})

    hard = x4.copy()
    hard = hard + local.normal(0.0, 0.18, size=hard.shape)
    hard = np.clip(hard, 0.0, 1.0)
    hard[:, ::3] = 0.65 * hard[:, ::3] + 0.35 * local.random(size=hard[:, ::3].shape)
    hard = np.roll(hard.reshape(-1, 8, 8), shift=1, axis=2).reshape(-1, 64)
    rungs.append({"name": "D5 harder real digits (shift+noise)", "X": hard[:900], "y": y4[:900], "kind": "image", "image_shape": (8, 8)})

    return rungs


def generated_baseline(rung, seed=SEED):
    local = np.random.default_rng(seed)
    X = np.asarray(rung["X"], dtype=float)
    if X.shape[1] == 1:
        return local.normal(X.mean(axis=0), X.std(axis=0) + 1e-6, size=X.shape)
    idx = local.choice(len(X), size=len(X), replace=True)
    boot = X[idx].copy()
    noise = local.normal(0.0, 0.10 * (X.std(axis=0) + 1e-6), size=boot.shape)
    return boot + noise


def feature_matrix(X):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    if X.shape[1] > 10:
        return pca_features(X, n_components=10)
    return standardize_features(X)


def sqrtm_psd(A):
    vals, vecs = np.linalg.eigh((A + A.T) / 2.0)
    vals = np.clip(vals, 0.0, None)
    return (vecs * np.sqrt(vals)) @ vecs.T


def fid_distance(real, gen):
    R = feature_matrix(real)
    G = feature_matrix(gen)
    mr = R.mean(axis=0)
    mg = G.mean(axis=0)
    cr = np.cov(R, rowvar=False) + np.eye(R.shape[1]) * 1e-6
    cg = np.cov(G, rowvar=False) + np.eye(G.shape[1]) * 1e-6
    middle = sqrtm_psd(sqrtm_psd(cr) @ cg @ sqrtm_psd(cr))
    return float(np.sum((mr - mg) ** 2) + np.trace(cr + cg - 2.0 * middle))


def two_sample_distance(real, gen):
    return fid_distance(real, gen)


def gaussian_nll(real, gen=None):
    X = feature_matrix(real)
    mu = X.mean(axis=0, keepdims=True)
    var = X.var(axis=0, keepdims=True) + 1e-4
    logp = -0.5 * (((X - mu) ** 2) / var + np.log(2.0 * np.pi * var))
    return float(-logp.sum(axis=1).mean())


def show_artifact(ax, X, rung, title):
    X = np.asarray(X)
    if rung.get("kind") == "image":
        img = X[0].reshape(rung.get("image_shape", (8, 8)))
        ax.imshow(img, cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    elif X.shape[1] == 1:
        ax.hist(X[:, 0], bins=24, color="steelblue", alpha=0.8)
    else:
        ax.scatter(X[:, 0], X[:, 1], s=8, c=rung.get("y"), cmap="tab10", alpha=0.75)
        ax.set_xticks([])
        ax.set_yticks([])
    ax.set_title(title, fontsize=9)


def plot_generated_panels(rungs, generated, metrics, ylabel):
    fig, axes = plt.subplots(2, len(rungs), figsize=(3.0 * len(rungs), 5.2))
    for j, rung in enumerate(rungs):
        show_artifact(axes[0, j], rung["X"], rung, "real " + rung["name"][:16])
        show_artifact(axes[1, j], generated[j], rung, "generated")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(np.arange(1, len(metrics) + 1), metrics, marker="o")
    ax.set_xlabel("rung D1 to D5")
    ax.set_ylabel(ylabel)
    ax.set_title("Metric curve across the F9 ladder")
    ax.grid(True, alpha=0.3)
    plt.show()


def preview_ladder(rungs):
    for i, rung in enumerate(rungs, start=1):
        X = rung["X"]
        y = rung.get("y")
        labels = "none" if y is None else np.unique(y)[:10].tolist()
        print(f"D{i}: {rung['name']}: X={X.shape}, labels/classes={labels}")


class GaussianEnergyModel:
    def fit(self, X):
        F = feature_matrix(X)
        self.mu = F.mean(axis=0)
        self.var = F.var(axis=0) + 0.05
        self.dim = F.shape[1]
        return self

    def energy(self, X):
        F = np.asarray(X, dtype=float)
        return 0.5 * np.sum(((F - self.mu) ** 2) / self.var, axis=1)

    def grad_energy(self, X):
        F = np.asarray(X, dtype=float)
        return (F - self.mu) / self.var

    def langevin(self, n, steps=80, step_size=0.03, seed=SEED):
        local = np.random.default_rng(seed)
        x = local.normal(size=(n, self.dim))
        for _ in range(steps):
            noise = local.normal(size=x.shape)
            x = x - step_size * self.grad_energy(x) + np.sqrt(2.0 * step_size) * noise
        return x


## The concept, built once: energy to probability
An EBM defines an unnormalized probability $$p_\theta(x)=\exp(-E_\theta(x))/Z_\theta,$$ where $$Z_\theta=\int \exp(-E_\theta(x))dx.$$ On a finite toy state space we can compute the partition function exactly.

In [ ]:
def energy_to_probability_and_sampler(energies):
    energies = np.asarray(energies, dtype=float)
    weights = np.exp(-energies)
    Z = weights.sum()
    probs = weights / Z
    return float(Z), probs

Z, probs = energy_to_probability_and_sampler(np.array([0.2, 1.1, 2.0]))
print(Z, probs[0])
assert round(Z, 3) == 1.287
assert round(float(probs[0]), 4) == 0.6362

For continuous data, exact $Z$ is usually unavailable, so sampling uses energy gradients. Langevin dynamics alternates downhill motion with noise so the model explores instead of only descending to one point.

In [ ]:
def fit_energy_and_sample(X, seed=SEED):
    model = GaussianEnergyModel().fit(X)
    samples = model.langevin(len(X), seed=seed)
    nll_proxy = float(model.energy(feature_matrix(X)).mean())
    return samples, nll_proxy, model

## The dataset ladder
We use the same F9 ladder inline in every notebook: 1-D Gaussian, 2-D moons, a three-component mixture, real sklearn digits, and a harder no-download real-digits rung with shift plus noise. This keeps the CPU path deterministic while increasing sample complexity.

In [ ]:
rungs = make_f9_ladder()
preview_ladder(rungs)

fig, axes = plt.subplots(1, len(rungs), figsize=(14, 2.6))
for ax, rung in zip(axes, rungs):
    show_artifact(ax, rung["X"], rung, rung["name"])
plt.tight_layout()
plt.show()

In [ ]:
metrics = []
generated = []
for rung in rungs:
    samples, proxy, model = fit_energy_and_sample(rung["X"])
    generated.append(samples)
    metric = proxy + 0.1 * two_sample_distance(feature_matrix(rung["X"]), samples)
    metrics.append(metric)
    print(f"{rung['name']:<34} energy/NLL proxy={metric:8.4f}")

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    plot_generated_panels(rungs, generated, metrics, "energy-ranked 2-sample distance")
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Pitfall on D5: weak negative samples
Weak negatives teach the EBM only where the sampler visits. A replay buffer plus longer Langevin chains visits harder negatives and raises their energy more reliably.

In [ ]:
d5 = rungs[-1]
model = GaussianEnergyModel().fit(d5["X"])
weak = model.langevin(200, steps=3, step_size=0.005, seed=SEED + 1)
strong = model.langevin(200, steps=120, step_size=0.025, seed=SEED + 2)
real_e = model.energy(feature_matrix(d5["X"])[:200]).mean()
weak_e = model.energy(weak).mean()
strong_e = model.energy(strong).mean()
print(f"real energy={real_e:.3f}")
print(f"weak negative energy={weak_e:.3f}")
print(f"replay/long-chain negative energy={strong_e:.3f}")

## Evaluate it + Practice
- Report `energy-ranked 2-sample distance` beside a no-skill baseline that resamples training examples with small Gaussian jitter.
- Cheap sanity check: generated panels should have the same support and rough diversity as the real panels.
- Ablation: use only three Langevin steps for negatives; the metric should get worse.
- Failure signal: D5 can look plausible in one panel while the summary curve exposes collapse or oversmoothing.
- Re-run with a different seed only after the structural logic is correct.

Practice prompts:
1. Change the D3 mixture weights and predict which metric moves most.
2. Add a bootstrap interval for D5 before trusting a small metric gap.
3. Replace PCA features with raw features on D4/D5 and explain the tradeoff.